# LLM Evaluation on Greek Protipa Exams

This notebook evaluates multiple LLMs on the `PennyK98/protipa_exams_dataset` from Hugging Face. It focuses on Greek Language and Mathematics multiple-choice questions.

In [40]:
import json
import logging
import requests
import os
import random
import time
import traceback
from pathlib import Path

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset
from dotenv import load_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [41]:
load_dotenv()

# API Config
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("LITELLM_HOST")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-01-09 13:06:40 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


## 2. Load and Prepare Dataset

Παράδειγμα

In [ ]:
logger.info("Loading dataset...")
original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")

#Ελέγχουμε μόνο τα Μαθηματικά και τη Γλώσσα και μόνο ερωτήσεις τύπου Multiple Choice χωρίς εικόνες
def filter_dataset(dataset):
    filtered = []
    subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ']
    for item in dataset:
        multimodal_status = item.get('multimodality')
        if (item['subject'] in subjects and 
            item['exercise_type'] == 'Multiple Choice' and 
            ',' not in str(item['answer_index']) and multimodal_status == 'no'):
            filtered.append(item)
    return filtered

all_filtered = filter_dataset(original_dataset)
logger.info(f"Filtered dataset size: {len(all_filtered)}")

# Sample for pilot test
pilot_100 = random.sample(all_filtered, min(len(all_filtered), 100))

# Save temporary JSON for lm_eval ingestion
json_path = (results_dir / "pilot_data_test.json").resolve()
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(pilot_100, f, ensure_ascii=False, indent=4)

logger.info(f"Pilot data (100 samples) saved to: {json_path}")

2026-01-09 09:36:01 - INFO - Loading dataset...
2026-01-09 09:36:05 - INFO - Filtered dataset size: 192
2026-01-09 09:36:05 - INFO - Pilot data (100 samples) saved to: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\tmp\pilot_data_test.json


RQ1 & RQ3

In [42]:
logger.info("Loading dataset...")
original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")

def filter_dataset(dataset):
    filtered = []
    
    subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ', 'ΦΥΣΙΚΗ', 'ΘΡΗΣΚΕΥΤΙΚΑ']
    
    target_types = ['Multiple Choice', 'True/False', 'Fill-in-the-gaps', 'Matching'] 
    
    for item in dataset:
        subj = item.get('subject')
        q_type = item.get('question_type')
        m_status = item.get('multimodality')
        choices = item.get('choices')

        if (subj in subjects and 
            q_type == 'Closed' and          
            m_status == 'no' and            
            item['exercise_type'] in target_types and
            choices is not None and         
            len(choices) > 1):              
            
            filtered.append(item)
            
    return filtered

all_filtered = filter_dataset(original_dataset)

#logger.info(f"Filtered dataset size for FULL RQ1: {len(all_filtered)}")
logger.info(f"Filtered dataset size for FULL RQ3: {len(all_filtered)}")

final_data = all_filtered

# Save JSON 
#json_path = (results_dir / "rq1_all_subjects_closed_types.json").resolve()
json_path = (results_dir / "rq3_closed_types_data.json").resolve()
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

logger.info(f"Data saved to: {json_path}")

2026-01-09 13:06:56 - INFO - Loading dataset...
2026-01-09 13:06:58 - INFO - Filtered dataset size for FULL RQ3: 227
2026-01-09 13:06:58 - INFO - Data saved to: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\tmp\rq3_closed_types_data.json


## 3. Define Evaluation Task Template

Task_config για ερωτήσεις κλειστού τύπου

In [43]:
task_config = {
    "task": "greek_protipa_exams",
    "dataset_path": "json",
    "num_fewshot": 3,      # Added few-shot parameter
    "dataset_kwargs": {
        "data_files": str(json_path)  # FIXED: Convert Path object to string
    },
    "test_split": "train",
    "output_type": "generate_until",
    "doc_to_text": (
        "{% if input %}{{input}}\n{% endif %}"
        "Ερώτηση: {{question}}\n"
        "Επιλογές:\n"
        "{% for choice in choices %}"
        "{{loop.index0}}. {{choice}}\n"
        "{% endfor %}\n"
        "### ΟΔΗΓΙΑ ΜΟΡΦΟΠΟΙΗΣΗΣ (CRITICAL)\n"
        "Πρέπει να παρέχεις ΜΟΝΟ τον αριθμό του δείκτη (index) της σωστής επιλογής.\n"
        "Μην επεξηγείς και μην γράφεις ολόκληρες προτάσεις.\n\n"
        "❌ ΛΑΘΟΣ: \"Η σωστή επιλογή είναι η 2.\"\n"
        "❌ ΛΑΘΟΣ: \"Ας υποθέσουμε ότι το κλάσμα είναι 1/12, άρα η απάντηση είναι 2.\"\n"
        "❌ ΛΑΘΟΣ: \"(2)\"\n"
        "❌ ΛΑΘΟΣ: \"2.\"\n"
        "✅ ΣΩΣΤΟ: 2\n\n"
        "Απάντηση: "
    ),
    "doc_to_target": "{{ (answer_index | string).split(',')[0] }}",
    "generation_kwargs": {
        "until": ["\n"],
        "max_gen_toks": 50,
        "do_sample": False,
        "temperature": 0.0 
    },
    "filter_list": [
        {
            "name": "strict-match",
            "filter": [
                {"function": "regex", "regex_pattern": r"(?<![0-9/])([0-9])(?![0-9/])"},
                {"function": "take_first"}
            ]
        }
    ],
    "metric_list": [
        {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
    ]
}

# Workaround for Task Config loading
task_dir = results_dir
task_dir.mkdir(parents=True, exist_ok=True)
with open(task_dir / "greek_protipa.yaml", "w", encoding='utf-8') as f:
    yaml.dump(task_config, f, allow_unicode=True)

custom_task = ConfigurableTask(config=task_config)
task_dict = {"greek_protipa_exams": custom_task}
logger.info("Evaluation task defined successfully.")

Generating train split: 0 examples [00:00, ? examples/s]

2026-01-09 13:07:08 - WARNING - [Task: greek_protipa_exams] num_fewshot > 0 but fewshot_split is None. using preconfigured rule.
2026-01-09 13:07:08 - WARNING - [Task: greek_protipa_exams] num_fewshot > 0 but fewshot_split is None. using preconfigured rule.
2026-01-09 13:07:08 - INFO - Evaluation task defined successfully.


## 4. Run Evaluation

In [44]:
comparison_results = {}
all_samples = {}

EVAL_LIMIT = None  # Adjust this to run more/less samples

for model_name in models_to_test:
    logger.info(f"Starting evaluation for model: {model_name}")
    try:
        # Construct endpoint URL
        chat_api_url = api_base
        if not chat_api_url.endswith("/chat/completions"):
            chat_api_url = chat_api_url.rstrip("/") + "/chat/completions"

        model = OpenAIChatCompletion(
            model=model_name,
            base_url=chat_api_url,
            num_fewshot=0,
            eos_string="<|end_of_text|>",
            max_retries=10,
            num_concurrent=1
        )

        results = lm_eval.evaluate(
            lm=model,
            task_dict=task_dict,
            limit=EVAL_LIMIT,
            apply_chat_template=True
        )
        
        # 1. Store Summary Metrics
        scores = results['results']['greek_protipa_exams']
        comparison_results[model_name] = scores
        
        # 2. Collect Individual Samples for the table
        if 'samples' in results and 'greek_protipa_exams' in results['samples']:
            samples = results['samples']['greek_protipa_exams']
            for i, s in enumerate(samples):
                if i not in all_samples:
                    doc = s.get('doc', {})
                    all_samples[i] = {
                        "Question": doc.get('question', 'N/A'),
                        "Subject": doc.get('subject', 'N/A'),
                        "Exercise Type": doc.get('exercise_type', 'N/A'),
                        "Year": doc.get('year', 'N/A'),
                        "Level": doc.get('school_level', 'N/A'),
                        "Ground Truth": str(doc.get('answer_index', 'N/A')),
                        "Choices": " | ".join(doc.get('choices', [])),
                        "Model Predictions": {},
                        "Raw Responses": {}
                    }
                
                all_samples[i]["Model Predictions"][model_name] = s.get('filtered_resps', ["N/A"])[0]
                all_samples[i]["Raw Responses"][model_name] = s.get('resps', [["N/A"]])[0][0]

        acc = scores.get('exact_match,strict-match', scores.get('acc', 0.0))
        logger.info(f"Success! {model_name} Accuracy: {acc:.2%}")
        
        logger.info("Waiting 2 seconds before next model to avoid rate-limits...")
        time.sleep(2)

    except Exception as e:
        logger.error(f"Error evaluating {model_name}: {e}")
        logger.error(traceback.format_exc())
        time.sleep(2)

2026-01-09 13:07:24 - INFO - Starting evaluation for model: gemma3-27b-it
2026-01-09 13:07:24 - INFO - Using max length 2048 - 1
2026-01-09 13:07:24 - INFO - Concurrent requests are disabled. To enable concurrent requests, set `num_concurrent` > 1.
2026-01-09 13:07:24 - INFO - Using tokenizer None
2026-01-09 13:07:24 - WARNING - Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
2026-01-09 13:07:24 - INFO - Building contexts for greek_protipa_exams on rank 0...
100%|██████████| 227/227 [00:02<00:00, 80.48it/s] 
2026-01-09 13:07:26 - INFO - Running generate_until requests
2026-01-09 13:07:26 - INFO - Tokenized requests are disabled. Context + generation length is not checked.
Requesting API: 100%|██████████| 227/227 [06:50<00:00,  1.81s/it]
2026-01-09 13:14:18 - INFO - Success! gemma3-27b-it Accuracy: 57.71%
2026-01-09 13:14:18 - INFO - Waiting 2 seconds before next model to avoid rate-limits...
2026-01-09 13:14

## 5. Results Table

In [45]:
if all_samples:
    table_data = []
    for idx, data in all_samples.items():
        row = {
            "ID": idx,
            "Subject": data["Subject"],
            "Exercise Type": data.get("Exercise Type", "N/A"),
            "Year": data["Year"],
            "Level": data["Level"],
            "Question": data["Question"],
            "Choices": data["Choices"],
            "Ground Truth": data["Ground Truth"]
        }
        for m in models_to_test:
            # Add both the extracted prediction AND the raw text from the model
            row[f"{m}_pred"] = data["Model Predictions"].get(m, "N/A")
            # row[f"{m}_raw"] = data["Raw Responses"].get(m, "N/A") 
            
        table_data.append(row)
    
    df_results = pd.DataFrame(table_data)
    
    # Save CSV
    #results_file = results_dir / "eval_results_table.csv"
    results_file = results_dir/ "eval_results_RQ3_closed_types_table.csv"
    #results_file = results_dir/ "eval_results_RQ1_all_subjects_closed_types_table.csv"
    df_results.to_csv(results_file, index=False, encoding='utf-8-sig')
    logger.info(f"Table saved to: {results_file}")
    
    # This will now display the raw responses in the table below
    display(df_results.head(10)) 

2026-01-09 13:19:50 - INFO - Table saved to: tmp\eval_results_RQ3_closed_types_table.csv


,ID,Subject,Exercise Type,Year,Level,Question,Choices,Ground Truth,gemma3-27b-it_pred,krikri-dpo-context_pred
0,0,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,2020,ΓΥΜΝΑΣΙΟ,"Στην αριθμογραμμή, ο αριθμός που βρίσκεται ακρ...",A. $1/8$ | B. $2/8$ | Γ. $16/63$ | Δ. $8/63$,3,2,2
1,1,ΓΛΩΣΣΑ,Multiple Choice,2025,ΛΥΚΕΙΟ,"Ποιο από τα παρακάτω δεν ισχύει, σύμφωνα με το...",Α. Υπάρχει μόνο μία επιλογή προκειμένου να αντ...,0,0,2
2,2,ΓΛΩΣΣΑ,Multiple Choice,2022,ΛΥΚΕΙΟ,«εφικτός»: Ποια λέξη έχει αντίθετη σημασία;,Α. ακατόρθωτος | Β. προαιρετικός | Γ. ανεπιτυχ...,0,0,0
3,3,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,2023,ΛΥΚΕΙΟ,Η εξίσωση $\frac{1}{2}-\frac{x-2}{4}=1$ έχει α...,A. $2-x+2=4$ | B. $.2-x+2=1$ | Γ. $-2-x-2=1$ |...,0,0,2
4,4,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,2025,ΓΥΜΝΑΣΙΟ,Αναμειγνύουμε ίδια ποσότητα από τρία ροφήματα....,A. 22% | B. 26% | Γ. 28% | Δ. 30%,1,2,2
5,5,ΓΛΩΣΣΑ,Multiple Choice,2020,ΛΥΚΕΙΟ,Ο σκοπός του Αναγνωστόπουλου ήταν:,Α. να αποτυπώσει την απλότητα της φύσης. | Β. ...,0,0,0
6,6,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,2023,ΓΥΜΝΑΣΙΟ,"Έχουμε τρία ίδια ποτήρια με νερό. Αρχικά, το $...",A. $\frac{1}{2}$ | B. $1$ | Γ. $\frac{3}{8}$ |...,2,3,0
7,7,ΓΛΩΣΣΑ,Multiple Choice,2021,ΓΥΜΝΑΣΙΟ,Ποια από τις παρακάτω φράσεις ΔΕΝ ταιριάζει με...,Α. μας φώναξε | Β. μας έβαλε | Γ. μας περιποιό...,3,3,2
8,8,ΘΡΗΣΚΕΥΤΙΚΑ,Multiple Choice,2025,ΛΥΚΕΙΟ,Η πιο γνωστή παράσταση του λειτουργικού εικονο...,Α. η Κοινωνία του Χριστού. | Β. το θαύμα στο γ...,2,2,2
9,9,ΜΑΘΗΜΑΤΙΚΑ,Multiple Choice,2022,ΓΥΜΝΑΣΙΟ,Η καθηγήτρια της Ιστορίας μας έχει βάλει να κά...,A. 23 | B. 22 | Γ. 21 | Δ. 20,0,0,0


## 6. Performance Summary

In [46]:
if comparison_results:
    summary_data = []
    for model, metrics in comparison_results.items():
        acc = metrics.get('exact_match,strict-match', metrics.get('acc', 0.0))
        summary_data.append({"Model": model, "Accuracy": acc})
    
    df_summary = pd.DataFrame(summary_data)
    display(df_summary)

,Model,Accuracy
0,gemma3-27b-it,0.577093
1,krikri-dpo-context,0.374449


In [47]:
#csv_path = results_dir / "eval_results_RQ1_all_subjects_closed_types_table.csv" 
csv_path = results_dir / "eval_results_RQ3_closed_types_table.csv"
df = pd.read_csv(csv_path)

print(f"📊 Φορτώθηκαν {len(df)} ερωτήσεις για ανάλυση.\n")

def normalize_val(val):
    """Καθαρίζει την τιμή για να γίνει σωστή σύγκριση."""
    s = str(val).strip()  # Αφαιρεί κενά και \n στην αρχή/τέλος
    # Αν κατά λάθος έγινε 1.0 (float string), το κάνουμε 1
    if s.endswith(".0"):
        s = s[:-2]
    return s

# Υπολογισμός Σωστού/Λάθους με τη νέα έξυπνη συνάρτηση
for model in ["gemma3-27b-it", "krikri-dpo-context"]:
    # Εφαρμόζουμε το καθάρισμα (normalize) και στις δύο στήλες
    gt_clean = df["Ground Truth"].apply(normalize_val)
    pred_clean = df[f"{model}_pred"].apply(normalize_val)
    
    # Τώρα συγκρίνουμε τα καθαρά δεδομένα
    df[f"{model}_correct"] = (gt_clean == pred_clean).astype(int)

📊 Φορτώθηκαν 227 ερωτήσεις για ανάλυση.



Ανάλυση RQ1: Ακρίβεια ανά μάθημα (κλειστού τύπου)

In [ ]:
# Ανάλυση RQ1: Ακρίβεια ανά Είδος Άσκησης
print("\n" + "-" * 60)
print("🏆 RQ1: PERFORMANCE PER SUBJECT")
print("-" * 60)

rq1_stats = df.groupby("Subject")[[
    "gemma3-27b-it_correct", 
    "krikri-dpo-context_correct"
]].agg(['mean', 'count'])

# Μορφοποίηση του πίνακα
rq1_final = rq1_stats.copy()
rq1_final.columns = ['Gemma Acc', 'Count', 'Krikri Acc', 'Count_duplicate']
rq1_final = rq1_final[['Gemma Acc', 'Krikri Acc', 'Count']]
rq1_final['Gemma Acc'] = (rq1_final['Gemma Acc'] * 100).round(1).astype(str) + '%'
rq1_final['Krikri Acc'] = (rq1_final['Krikri Acc'] * 100).round(1).astype(str) + '%'
display(rq1_final)
logger.info("Evaluation completed.")

Ανάλυση RQ3: Ακρίβεια ανά είδος άσκησης (κλειστού τύπου)

In [48]:
# Ανάλυση RQ3: Ακρίβεια ανά Είδος Άσκησης
print("\n" + "-" * 60)
print("🏆 RQ3: PERFORMANCE PER CLOSED EXERCISE TYPE")
print("-" * 60)

rq3_stats = df.groupby("Exercise Type")[[
    "gemma3-27b-it_correct", 
    "krikri-dpo-context_correct"
]].agg(['mean', 'count'])

# Μορφοποίηση του πίνακα
rq3_final = rq3_stats.copy()
rq3_final.columns = ['Gemma Acc', 'Count', 'Krikri Acc', 'Count_duplicate']
rq3_final = rq3_final[['Gemma Acc', 'Krikri Acc', 'Count']]
rq3_final['Gemma Acc'] = (rq3_final['Gemma Acc'] * 100).round(1).astype(str) + '%'
rq3_final['Krikri Acc'] = (rq3_final['Krikri Acc'] * 100).round(1).astype(str) + '%'
display(rq3_final)
logger.info("Evaluation completed.")


------------------------------------------------------------
🏆 RQ3: PERFORMANCE PER CLOSED EXERCISE TYPE
------------------------------------------------------------


,Gemma Acc,Krikri Acc,Count
Exercise Type,,,
Fill-in-the-gaps,100.0%,50.0%,2
Matching,0.0%,0.0%,3
Multiple Choice,56.4%,36.0%,211
True/False,90.9%,72.7%,11


2026-01-09 13:20:16 - INFO - Evaluation completed.


Διαθέσιμα μοντέλα

In [17]:
api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        # Φιλτράρουμε μόνο αυτά που μας ενδιαφέρουν
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            # Αν το ID περιέχει κάποια από τις λέξεις κλειδιά
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

[
    "mistral-large-24.02",
    "gpt-oss-120b",
    "mistral-small-24.02",
    "mistral-7b-instruct-v0.2",
    "llama-3.2-1b",
    "gemma3-27b-it",
    "llama-3.1-8b",
    "llama-3.3-70b",
    "gemma3-27b-it-long",
    "llama-3.2-3b",
    "gpt-4o-mini",
    "gpt-4o",
    "llama-3.1-70b",
    "gpt-oss-20b"
]


**ΠΑΡΑΤΗΡΗΣΕΙΣ**

○ Τα υπόλοιπα μοντέλα (πχ llama-3.1-8b, mistral-7b-instruct-v0.2) σκάνε με Bedrock error όταν τα τρέχω

***ΓΛΩΣΣΑ ΚΑΙ ΜΑΘΗΜΑΤΙΚΑ, multiple-choice questions*** (παράδειγμα)

○ Το accuracy του krikri και του gemma ήταν 29% και 48% αντίστοιχα όταν τα τρέχω με το multimodality == 'yes'

○ Θέτοντας το multimodality == 'no', πέφτει η ακρίβεια και των 2 μοντέλων σε 22% (krikri) και 43% (gemma)

○ Το krikri σε zero-shot setting καταρρέει και "κλειδώνει" ότι η σωστή απάντηση βρίσκεται πάντα στο index 2 σε όλες τις απαντήσεις που δίνει  

○ Αλλάζοντας τα settings του task_config σε "num_fewshot": 3, αυξάνονται τα ποσοστά της ακρίβειας σε 54% για το gemma και 40% για το krikri χωρίς να κλειδώνει στο index 2, δίνοντας κι άλλες θέσεις (0, 1, 3) για τη σωστή απάντηση

***ΟΛΑ ΤΑ ΜΑΘΗΜΑΤΑ, all closed-type questions*** (RQ1: Comparative Performance Across Subjects)

○ Το accuracy του krikri και του gemma βρίσκεται στο 41% και 57% αντίστοιχα με "num_fewshot": 3 settings

○ ΘΡΗΣΚΕΥΤΙΚΑ (Gemma 73.7% / Krikri 57.9%) - Ο "Νικητής"

○ ΓΛΩΣΣΑ (Gemma 68.3% / Krikri 45.5%) - Η "Μέση Οδός"

○ ΜΑΘΗΜΑΤΙΚΑ (Gemma 36.5% / Krikri 30.6%) - Ο "Μεγάλος Ασθενής"

***ΟΛΑ ΤΑ ΜΑΘΗΜΑΤΑ, all types of closed questions (multiple-choice, fill-in-the-gaps, true/false, matching)*** (RQ3: Performance Across Exercise Types)

○ Το accuracy του krikri και του gemma παραμένει περίπου στο 40% και 53% αντίστοιχα με "num_fewshot": 3 settings

○ Ο τωρινός κώδικας (task_config) είναι φτιαγμένος να ψάχνει για ένα index, δηλαδή έναν αριθμό (π.χ. 1, 2, 3). Αυτό δουλεύει στο Multiple Choice, στο True/False και στο Fill-in-the-gaps γιατί εκεί η σωστή απάντηση είναι "Επιλογή 1" ή "Επιλογή 2". Στο Matching, η απάντηση δεν είναι ένας αριθμός. Είναι μια λίστα με ζεύγη (["1-γ", "2-α", "3-ε", "4-β", "5-δ"]). Στο dataset, το πεδίο answer_index είναι κενό (None) γιατί δεν υπάρχει "μία σωστή επιλογή", αλλά ένας συνδυασμός. Καλό θα ήταν να εξαιρέσουμε το Matching από αυτό το πείραμα (Closed Types), καθώς χρειάζεται άλλο κώδικα αξιολόγησης και πρόκειται για ελάχιστα παραδείγματα.